# Parser et packetizer de conclusions juridiques françaises

## Objectif

Transformer un long document juridique en français (des conclusions, un mémoire ou une plaidoirie) en une structure exploitable par un LLM.

Le pipeline suit 2 grandes étapes :

1. **Parser le document** en sections logiques :
   - en-tête
   - faits
   - procédure
   - discussion
   - demandes / dispositif
   - autres sous-sections argumentatives

2. **Découper ces sections en paquets ("packets")** :
   - chaque paquet contient plusieurs sections consécutives,
   - le découpage respecte au mieux la logique argumentative,
   - le volume de texte reste compatible avec une fenêtre de contexte LLM.

Le script génère aussi :
- des métadonnées légères (articles, jurisprudences, dates, montants, entités),
- des prompts système et utilisateur pour brancher ensuite un LLM,
- des fichiers JSON de sortie permettant inspection et intégration.

---

## Vue d'ensemble du pipeline

### 1. Lecture du texte source

Le script lit un fichier `.txt` contenant le document juridique brut.

### 2. Détection des titres / sections

Le document est analysé ligne par ligne pour détecter :
- les titres de haut niveau (`I -`, `II -`, etc.),
- les sous-parties alphabétiques (`A -`, `B -`),
- les sections numérotées (`1.`, `2)`, etc.),
- certains titres en majuscules (`DISCUSSION`, `LES FAITS`, etc.).

Cette détection permet de reconstruire une hiérarchie logique.

### 3. Construction des `SectionNode`

Chaque section détectée devient un objet `SectionNode` contenant :
- son identifiant,
- son titre,
- son niveau hiérarchique,
- son texte,
- son type de section,
- son chemin hiérarchique (`path`),
- ses métadonnées.

### 4. Extraction de métadonnées légères

Pour chaque section, le script extrait automatiquement :
- des références à des articles,
- des références de jurisprudence,
- des montants en euros,
- des dates,
- des entités nommées :
  - personnes,
  - avocats,
  - associations,
  - sociétés,
  - juridictions,
  - acteurs institutionnels.

Un traitement spécifique de l'en-tête permet aussi d'identifier :
- les parties principales,
- leurs rôles procéduraux,
- leur avocat,
- leur position relative (partie courante / partie adverse).

### 5. Post-traitement des sections

Le script fusionne certains faux positifs de segmentation, par exemple :
- un titre isolé sans vrai contenu,
- une mini-section qui doit plutôt être rattachée à la suivante.

### 6. Construction des paquets (`Packet`)

Les sections sont regroupées en paquets successifs, en respectant :
- l'ordre du document,
- la continuité rhétorique,
- la limite de budget de tokens.

Si une section est trop volumineuse, elle est découpée :
- d'abord par paragraphes,
- sinon par phrases.

### 7. Génération de prompts pour LLM

Le script fournit deux familles de prompts :

#### a. Prompt d'extraction intermédiaire
Il sert à demander à un LLM de transformer un paquet en JSON structuré, sans encore produire le résumé final.

#### b. Prompt de synthèse finale
Il sert à demander à un LLM de rédiger un résumé juridique final à partir des JSON extraits sur chaque paquet.

### 8. Export des artefacts

Le script écrit dans un dossier :
- `parsed_nodes.json`
- `packets.json`
- `extraction_system_prompt.txt`
- `final_system_prompt.txt`
- un fichier prompt par paquet dans `packet_prompts/`

---

## Structures de données principales

### `SectionNode`

Représente une section logique du document.

Champs principaux :
- `id` : identifiant de section
- `title` : titre de la section
- `level` : niveau hiérarchique
- `text` : texte de la section
- `section_type` : type de section (`header`, `facts`, `procedure`, etc.)
- `path` : chemin hiérarchique complet
- `meta` : métadonnées extraites

### `Packet`

Représente un groupe de sections consécutives prêtes à être envoyées à un LLM.

Champs principaux :
- `id`
- `nodes`
- `total_tokens`
- `section_types`
- `titles`
- `context_ribbon` : résumé contextuel du paquet

---

## Logique détaillée par module

## 1. Heuristiques

Le script repose sur plusieurs familles d'heuristiques :

### `SECTION_TYPE_PATTERNS`
Associe des expressions régulières à des types de sections :
- faits
- procédure
- discussion
- demandes
- etc.

### `HEADING_PATTERNS`
Détecte les formats de titres :
- chiffres romains,
- lettres,
- nombres,
- griefs,
- lignes en majuscules.

### Patterns métiers
Des regex supplémentaires servent à détecter :
- articles,
- jurisprudence,
- montants,
- dates,
- entités nommées.

---

## 2. Utilitaires

Quelques fonctions utilitaires assurent :
- la normalisation des espaces,
- le nettoyage des lignes,
- l'estimation approximative du nombre de tokens,
- le nettoyage des références jurisprudentielles,
- la détermination d'un type de section.

---

## 3. Parsing du document

### `parse_document(text)`

Fonction centrale de parsing.

Elle :
1. découpe le texte en lignes,
2. détecte les titres,
3. crée les bornes de chaque section,
4. reconstitue la hiérarchie via une pile (`stack`),
5. enrichit chaque section avec des métadonnées.

Cas particulier :
- si aucun titre n'est détecté, tout le document devient une seule section.

---

## 4. Extraction de l'en-tête

### `extract_header_registry(text)`

Cette fonction analyse le haut du document pour récupérer :
- les parties,
- leur qualité procédurale,
- leur camp,
- leur avocat.

Elle se base sur des marqueurs comme :
- `POUR :`
- `CONTRE :`
- `APPELANT`
- `INTIMÉ`

Ces informations sont ensuite utilisées pour améliorer l'interprétation des entités dans le reste du document.

---

## 5. Extraction d'entités

### `extract_entities(text)`

Le script détecte des entités juridiques utiles :
- avocats,
- personnes,
- associations,
- sociétés,
- juridictions,
- médecin du travail, etc.

Pour chaque entité, il tente d'inférer :
- son rôle,
- son camp,
- puis de la rattacher si possible à une entité canonique issue de l'en-tête.

---

## 6. Packetization

### `build_packets(nodes, ...)`

Regroupe les sections en paquets compatibles avec un LLM.

Le budget réellement disponible pour le contenu est calculé comme :

```math
safe_content_budget = max_input_tokens - prompt_budget_tokens - output_budget_tokens
```

Le but est de :
- maximiser la quantité de contenu utile par paquet  
- éviter de couper au milieu d'une continuité argumentative  
- préserver l'ordre du document  

---

## Gestion des sections trop volumineuses

### `split_oversized_node(node, safe_content_budget)`

Si une section dépasse la limite :

- découpage par paragraphes  
- puis découpage par phrases en secours  

---

## 7. Génération des prompts

### `build_extraction_system_prompt()`

Définit le rôle du LLM pour l'extraction intermédiaire.

---

### `build_extraction_user_prompt(packet)`

Construit le prompt utilisateur pour un paquet donné, avec :

- contexte  
- métadonnées détectées  
- schéma JSON cible  
- texte du paquet  

---

### `build_final_system_prompt()`

Définit le rôle du LLM pour la synthèse finale.

---

### `build_final_user_prompt(extracted_packets_json, mode, max_pages_hint)`

Construit le prompt final selon le type de rendu attendu :

- résumé global  
- faits et procédure  
- moyens  
- rapport  
- exposé du litige  

---

## Utilisation en ligne de commande

### Commande

```bash
python script.py chemin/vers/document.txt
```

---

### Options disponibles

```bash
python script.py chemin/vers/document.txt \
  --max-input-tokens 42000 \
  --prompt-budget 2500 \
  --output-budget 3000 \
  --dump-dir artifacts
```

---

### Paramètres

- `input_path` : chemin du fichier texte source  
- `--max-input-tokens` : taille max de la fenêtre LLM  
- `--prompt-budget` : réserve pour le prompt  
- `--output-budget` : réserve pour la réponse du modèle  
- `--dump-dir` : dossier de sortie  

---

## Fichiers générés

Dans le dossier `artifacts/` :

- `parsed_nodes.json`  
  Liste complète des sections parsées  

- `packets.json`  
  Liste des paquets générés  

- `extraction_system_prompt.txt`  
  Prompt système pour l'extraction intermédiaire  

- `final_system_prompt.txt`  
  Prompt système pour la synthèse finale  

- `packet_prompts/P1.txt`, `P2.txt`, etc.  
  Un prompt utilisateur par paquet  

---

## Cas d’usage typique

### Étape 1

Convertir un PDF ou DOCX en texte brut.

### Étape 2

Lancer ce script pour :

- segmenter le document  
- générer les paquets  
- préparer les prompts  

### Étape 3

Envoyer chaque paquet à un LLM avec :

- `build_extraction_system_prompt()`  
- `build_extraction_user_prompt(packet)`  

### Étape 4

Récupérer les JSON intermédiaires.

### Étape 5

Envoyer tous les JSON au LLM final avec :

- `build_final_system_prompt()`  
- `build_final_user_prompt(...)`  

---

## Forces du script

- Simple à intégrer  
- Adapté aux écritures juridiques françaises  
- Préserve la structure argumentative  
- Produit des métadonnées utiles pour le suivi des entités  
- Compatible avec un pipeline RAG / summarization / extraction  

---

## Limites

- Le parser repose sur des heuristiques regex, donc il peut manquer certains formats atypiques  
- L'estimation de tokens est approximative  
- L'extraction des entités n'est pas un NER complet mais un système métier ciblé  
- Certains documents très bruités ou mal OCRisés peuvent réduire la qualité de segmentation  

---

## Pistes d'amélioration

- Ajouter un parseur plus robuste pour les documents OCR dégradés  
- Affiner les patterns de titres  
- Ajouter la détection des pièces citées  
- Ajouter une meilleure résolution de coréférences  
- Brancher un tokenizer réel pour estimer plus précisément les tokens  
- Générer automatiquement des tests unitaires à partir d'exemples réels  



# Application au cas d'usage de résumé de conclusions

Un cas d'usage cible en justice civile est le résumé des conclusions des appelants (demandeurs) et intimé (défenseurs).


## Analyse des verbatims testeurs

In [157]:
import pandas as pd

feedback = pd.read_csv("/home/jovyan/notebooks/tests_thomas/summarization/data/feedback_LLM.csv")

# remove names column
feedback = feedback.drop("nom testeur", axis=1)

# rename some columns expicitly
feedback.rename(columns={'résumé conclusions ddeur': 'résumé conclusions demandeur', 
                         'résumé conclu def': 'résumé conclusion défense',
                         'synthèse des faits proced et prétentions': 'synthèse des faits procedure et prétentions',
                         'synthèse des conclu brutes': 'synthèse des conclusions brutes',
                         'synthèse avec les 2 résumés de ccl ': 'synthèse avec les 2 résumés de conclusions'},
               inplace=True)


# print(feedback.head)
feedback.to_csv("/home/jovyan/notebooks/tests_thomas/summarization/data/feedback_anonymized.csv", index=False)

Import des notations.

In [158]:
# Notations data
notation = pd.read_csv("/home/jovyan/notebooks/tests_thomas/summarization/data/Notation_assistant.csv")

# print(notation.columns.tolist())

notation = notation.drop(columns=['Submission ID', 'Respondent ID', 'Submitted at'])
# print(notation.head())

notation.to_csv("/home/jovyan/notebooks/tests_thomas/summarization/data/notation_anonymized.csv", index=False)

## Script principal

In [159]:
from __future__ import annotations

import json
import math
import re
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple


# ============================================================
# Data structures
# ============================================================

@dataclass
class SectionNode:
    """
    Représente une section logique du document.

    Exemple :
    - I - LES FAITS
    - A - Sur le licenciement
    - 1. Sur l'absence de cause réelle et sérieuse

    Chaque section contient :
    - ses bornes dans le document,
    - son texte,
    - son type estimé,
    - son chemin hiérarchique,
    - des métadonnées légères extraites automatiquement.
    """
    id: str
    title: str
    level: int
    start_line: int
    end_line: int
    text: str
    section_type: str
    path: List[str] = field(default_factory=list)
    children: List["SectionNode"] = field(default_factory=list)
    meta: Dict[str, object] = field(default_factory=dict)

    @property
    def approx_tokens(self) -> int:
        """
        Estimation grossière du nombre de tokens.
        Sert pour le découpage en paquets compatibles avec la taille de contexte du LLM cible.
        """
        return approximate_tokens(self.text)


@dataclass
class Packet:
    """
    Représente un paquet de sections consécutives.
    L'idée est d'envoyer ce paquet à un LLM en respectant un budget max de tokens.
    """
    id: str
    nodes: List[SectionNode]
    total_tokens: int
    section_types: List[str]
    titles: List[str]
    context_ribbon: Dict[str, object]

    @property
    def text(self) -> str:
        """
        Construit une représentation textuelle du paquet avec le chemin hiérarchique de chaque section.
        """
        parts = []
        for node in self.nodes:
            path = " > ".join(node.path) if node.path else node.title
            parts.append(f"### PATH: {path}\n{node.text.strip()}")
        return "\n\n".join(parts)


# ============================================================
# Heuristics and constants
# ============================================================

# Associe des motifs de titres à un type de section métier.
SECTION_TYPE_PATTERNS: List[Tuple[str, List[re.Pattern[str]]]] = [
    (
        "header",
        [
            re.compile(r"\bPOUR\s*:", re.I),
            re.compile(r"\bCONTRE\s*:", re.I),
            re.compile(r"\bAPPELANT[E]?\b", re.I),
            re.compile(r"\bINTIM[ÉE]?\b", re.I),
        ],
    ),
    (
        "facts",
        [
            re.compile(r"\bLES FAITS\b", re.I),
            re.compile(r"\bRAPPEL DES FAITS\b", re.I),
            re.compile(r"\bEXPOS[ÉE] DES FAITS\b", re.I),
        ],
    ),
    (
        "procedure",
        [
            re.compile(r"\bLA PROC[ÉE]DURE\b", re.I),
            re.compile(r"\bRAPPEL DE LA PROC[ÉE]DURE\b", re.I),
            re.compile(r"\bPROC[ÉE]DURE\b", re.I),
        ],
    ),
    (
        "claims",
        [
            re.compile(r"\bPAR CES MOTIFS\b", re.I),
            re.compile(r"\bDISPOSITIF\b", re.I),
            re.compile(r"\bIL PLA[IÎ]T\b", re.I),
            re.compile(r"\bEN CONS[ÉE]QUENCE\b", re.I),
        ],
    ),
    (
        "discussion",
        [
            re.compile(r"\bDISCUSSION\b", re.I),
            re.compile(r"\bEN DROIT\b", re.I),
            re.compile(r"\bMOYENS\b", re.I),
            re.compile(r"\bARGUMENTAIRE\b", re.I),
        ],
    ),
]

# Motifs de titres classés du plus structurant au moins structurant.
HEADING_PATTERNS: List[Tuple[int, re.Pattern[str], str]] = [
    (1, re.compile(r"^\s*[IVXLCDM]+\s*[\.-]\s+.+$"), "roman"),
    (1, re.compile(r"^\s*[IVXLCDM]+\s*$"), "roman_only"),
    (2, re.compile(r"^\s*[A-Z]\s*-\s+.+$"), "alpha"),
    (3, re.compile(r"^\s*\d+\s*[-°\.)]\s+.+$"), "numeric"),
    (3, re.compile(r"^\s*GRIEF\s*\d+\s*:?\s*$", re.I), "grief"),
    (4, re.compile(r"^\s*[a-z]\)\s+.+$"), "subalpha"),
    # Uppercase topical lines like "DISCUSSION" or "A TITRE PRELIMINAIRE"
    (1, re.compile(r"^\s*[A-ZÉÈÀÙÂÊÎÔÛÇ'\-\s]{6,}\s*$"), "caps"),
]


ROMAN_HEADING_RE = re.compile(r"^\s*([IVXLCDM]+)\s*[\.-]\s+(.+?)\s*$")
ALPHA_HEADING_RE = re.compile(r"^\s*([A-Z])\s*[\.-]\s+(.+?)\s*$")
NUMERIC_HEADING_RE = re.compile(r"^\s*(\d+)\s*[-°\.)]\s+(.+?)\s*$")
GRIEF_RE = re.compile(r"^\s*(GRIEF\s*\d+)\s*:?\s*$", re.I)

# Regex de métadonnées juridiques
ARTICLE_RE = re.compile(r"\b(article|articles)\s+[A-Z]?\s*\d+[\d\-\.]*\b", re.I)
CASELAW_RE = re.compile(
    r"\bCass\.\s*"
    r"(?:soc\.|civ\.|com\.|crim\.)?"
    r".{0,80}?"
    r"n[°º]\s*\d{2}-\d{2}\.\d{3}\b",
    re.I,
)
MONEY_RE = re.compile(r"\b\d{1,3}(?:[ .]\d{3})*(?:,\d{2})?\s*€")
DATE_RE = re.compile(
    r"\b(?:\d{1,2}[./-]\d{1,2}[./-]\d{2,4}|\d{1,2}\s+[a-zéûîôàè]+\s+\d{4})\b",
    re.I,
)

# Patterns pour extraire certaines entités nommées spécifiques au domaine.
PERSON_ENTITY_PATTERNS: List[Tuple[str, re.Pattern[str], Optional[str], Optional[str]]] = [
    ("lawyer", re.compile(r"\bMa[iî]tre\s+\[[^\]]+\](?:\s+\[[^\]]+\])*", re.I), "neutral", "lawyer"),
    ("person", re.compile(r"\b(?:Monsieur|Madame|M\.)\s+\[[^\]]+\](?:\s+\[[^\]]+\])*", re.I), None, "person"),
    ("association", re.compile(r"\b(?:L['’])?ASSOCIATION\s+\[[^\]]+\](?:\s+\[[^\]]+\])*", re.I), None, "organization"),
    ("societe", re.compile(r"\b(?:Soci[ée]t[ée]|SARL|SAS|SA|SCI)\s+\[[^\]]+\](?:\s+\[[^\]]+\])*", re.I), None, "organization"),
    ("jurisdiction", re.compile(r"\b(?:Conseil de prud['’]hommes|Cour d['’]appel|Cour de cassation|Tribunal judiciaire)\b", re.I), "neutral", "institution"),
    ("occupational_physician", re.compile(r"\bm[ée]decin du travail\b", re.I), "neutral", "institutional_actor"),
]

# Indices lexicaux pour essayer d'inférer le rôle d'une entité
ROLE_HINT_PATTERNS: List[Tuple[re.Pattern[str], str]] = [
    (re.compile(r"\bappelant[e]?\b", re.I), "appelant"),
    (re.compile(r"\bintim[ée]?\b", re.I), "intimé"),
    (re.compile(r"\bdemandeur\b", re.I), "demandeur"),
    (re.compile(r"\bd[ée]fendeur\b", re.I), "défendeur"),
    (re.compile(r"\bvice-pr[ée]sident[e]?\b", re.I), "vice-président"),
    (re.compile(r"\bpr[ée]sident[e]?\b", re.I), "président"),
    (re.compile(r"\btr[ée]sorier\b|\btr[ée]sori[èe]re\b", re.I), "trésorier"),
    (re.compile(r"\bresponsable des ressources humaines\b|\br[ée]f[ée]rente RH\b", re.I), "RH"),
    (re.compile(r"\bsalari[ée]\b", re.I), "salarié"),
    (re.compile(r"\bemployeur\b", re.I), "employeur"),
]

# Indices lexicaux pour déterminer le "camp" de l'entité
SIDE_HINT_PATTERNS: List[Tuple[re.Pattern[str], str]] = [
    (re.compile(r"\bappelant[e]?\b", re.I), "current_party"),
    (re.compile(r"\bintim[ée]?\b", re.I), "opposing_party"),
    (re.compile(r"\bdemandeur\b", re.I), "current_party"),
    (re.compile(r"\bd[ée]fendeur\b", re.I), "opposing_party"),
    (re.compile(r"\bemployeur\b", re.I), "current_party"),
    (re.compile(r"\bsalari[ée]\b", re.I), "opposing_party"),
]

ROLE_BLACKLIST_KINDS = {"institution", "institutional_actor"}
ROLE_WINDOW = 80
SIDE_WINDOW = 80


# ============================================================
# Utility functions
# ============================================================


def approximate_tokens(text: str) -> int:
    """Approximation simple et indépendante du modèle (1 token pour 4 caractères)."""
    return max(1, math.ceil(len(text) / 4))



def normalize_space(text: str) -> str:
    """Normalise les espaces et les retours à la ligne 
    pour éviter les variations liées à l'OCR ou au copier-coller."""
    text = text.replace("\u00A0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()



def clean_line(line: str) -> str:
    """Réduit les espaces multiples dans une ligne"""
    return re.sub(r"\s+", " ", line).strip()


def clean_caselaw_match(s: str) -> str:
    """Nettoie une référence de jurisprudence pour éviter de conserver du bruit après le numéro d'arrêt."""
    s = clean_line(s)
    s = re.split(r"(?<=\d{2}-\d{2}\.\d{3})", s, maxsplit=1)[0]
    return s.strip(" .;,:)\"]»")


def guess_section_type(title: str, inherited: Optional[str] = None) -> str:
    """
    Devine le type d'une section à partir de son titre.
    Si rien n'est reconnu, on hérite du type précédent si possible,
    sinon on considère la section comme argumentaire générique.
    """
    for section_type, patterns in SECTION_TYPE_PATTERNS:
        if any(p.search(title) for p in patterns):
            return section_type
    if inherited:
        return inherited
    return "argument"



def heading_level_and_kind(line: str) -> Optional[Tuple[int, str]]:
    """
    Détermine si une ligne ressemble à un titre, et si oui :
    - son niveau hiérarchique,
    - son "kind" (roman, alpha, numeric, etc.)
    """
    stripped = clean_line(line)
    if not stripped:
        return None

    # Évite de confondre certaines lignes comme "M. X ..." avec un titre.
    if re.match(r"^(M|Mme|Mlle|Dr|Pr)\.\s+", stripped, re.I):
        return None

    for level, pattern, kind in HEADING_PATTERNS:
        if pattern.match(stripped):
            # Les lignes tout en majuscules trop longues sont probablement du texte normal (à confirmer).
            if kind == "caps" and len(stripped.split()) > 12:
                return None
            return level, kind
    return None


def split_lines(text: str) -> List[str]:
    """Uniformise les sauts de ligne puis découpe en liste de lignes."""
    return text.replace("\r\n", "\n").replace("\r", "\n").split("\n")



def extract_header_registry(text: str) -> Dict[str, object]:
    """
    Extrait les parties principales et leurs avocats à partir de l'en-tête.

    On cherche une structure de type :
    - POUR : ... APPELANTE ... Maître X
    - CONTRE : ... INTIME ... Maître Y

    Cela permet ensuite de recaler les entités détectées dans le document
    sur une référence canonique.
    """
    normalized = normalize_space(text)

    registry = {
        "document_view": None,   # e.g. appelant / intimé
        "parties": [],
        "by_name": {},
    }

    # Déduit éventuellement depuis quel point de vue le document est rédigé.
    if re.search(r"\bCONCLUSIONS D['’]APPELANT\b", normalized, re.I):
        registry["document_view"] = "appelant"
    elif re.search(r"\bCONCLUSIONS D['’]INTIM[ÉE]?\b", normalized, re.I):
        registry["document_view"] = "intimé"

    # Split autour de POUR / CONTRE
    header_text = normalized

    # On limite le parsing de l'en-tête au début du document
    cut_markers = [
        r"\bI[\.\-]\s+LES FAITS\b",
        r"\bLES FAITS\b",
        r"\bRAPPEL DES FAITS\b",
        r"\bDISCUSSION\b",
    ]
    for marker in cut_markers:
        m = re.search(marker, header_text, re.I)
        if m:
            header_text = header_text[:m.start()]
            break
    
    pour_match = re.search(r"\bPOUR\s*:\s*(.*?)(?=\bCONTRE\s*:|$)", header_text, re.I | re.S)
    contre_match = re.search(r"\bCONTRE\s*:\s*(.*?)(?=$)", header_text, re.I | re.S)


    blocks = []
    if pour_match:
        blocks.append(("pour", pour_match.group(1)))
    if contre_match:
        blocks.append(("contre", contre_match.group(1)))

    for label, block in blocks:
        party_name = None
        party_kind = None
        procedural_role = None
        side = None
        lawyer = None

        # On tente d'abord de reconnaître une organisation, sinon une personne.
        m_org = re.search(r"\b(?:L['’])?ASSOCIATION\s+\[[^\]]+\](?:\s+\[[^\]]+\])*", block, re.I)
        m_person = re.search(r"\b(?:Monsieur|Madame|M\.)\s+\[[^\]]+\](?:\s+\[[^\]]+\])*", block, re.I)

        if m_org:
            party_name = clean_line(m_org.group(0))
            party_name = re.sub(r"^(?:L['’])?ASSOCIATION\b", "Association", party_name, flags=re.I)
            party_kind = "organization"
        elif m_person:
            party_name = clean_line(m_person.group(0))
            party_kind = "person"

        # Rôle procédural trouvé dans le bloc.
        if re.search(r"\bAPPELANT[E]?\b", block, re.I):
            procedural_role = "appelant"
        elif re.search(r"\bINTIM[ÉE]?\b", block, re.I):
            procedural_role = "intimé"
        elif re.search(r"\bDEMANDEUR\b", block, re.I):
            procedural_role = "demandeur"
        elif re.search(r"\bD[ÉE]FENDEUR\b", block, re.I):
            procedural_role = "défendeur"

        # Déduction du camp à partir du rôle ou du bloc POUR / CONTRE.
        if procedural_role == "appelant":
            side = "current_party" if registry["document_view"] == "appelant" else "opposing_party"
        elif procedural_role == "intimé":
            side = "current_party" if registry["document_view"] == "intimé" else "opposing_party"
        elif label == "pour":
            side = "current_party"
        elif label == "contre":
            side = "opposing_party"

        # Recherche d'un avocat
        m_lawyer = re.search(r"\bMa[iî]tre\s+\[[^\]]+\](?:\s+\[[^\]]+\])*", block, re.I)
        if m_lawyer:
            lawyer = clean_line(m_lawyer.group(0))

        if party_name:
            entry = {
                "name": party_name,
                "kind": party_kind,
                "procedural_role": procedural_role,
                "side": side,
                "lawyer": lawyer,
            }
            registry["parties"].append(entry)
            registry["by_name"][party_name.lower()] = entry

            # On indexe aussi l'avocat dans le registre si trouvé.
            if lawyer:
                registry["by_name"][lawyer.lower()] = {
                    "name": lawyer,
                    "kind": "lawyer",
                    "procedural_role": procedural_role,
                    "side": side,
                    "represents": party_name,
                }

    return registry


def canonicalize_entity_against_registry(entity: Dict[str, Optional[str]], header_registry: Dict[str, object]):
    """
    Essaie de rattacher une entité trouvée dans le corps du texte
    à une entité canonique déjà extraite dans l'en-tête.
    """
    by_name = header_registry.get("by_name", {})
    name = entity["name"]
    low = name.lower()

    # exact match
    if low in by_name:
        ref = by_name[low]
        if ref.get("side"):
            entity["side"] = ref["side"]
        if ref.get("procedural_role"):
            entity["role"] = ref["procedural_role"]
        entity["canonical_name"] = ref.get("name")
        return entity

    # Cas plus souple : comparaison par contenus entre crochets, ex : "Monsieur [G]" vs "Monsieur [B] [G]"
    for ref_name, ref in by_name.items():
        if entity["kind"] != ref.get("kind"):
            continue

        # compare les tokens entre crochets (A MODIFIER POUR DOSSIERS NON ANONYMISES)
        entity_tokens = re.findall(r"\[[^\]]+\]", low)
        ref_tokens = re.findall(r"\[[^\]]+\]", ref_name)

        if entity_tokens and ref_tokens and set(entity_tokens).issubset(set(ref_tokens)):
            if ref.get("side"):
                entity["side"] = ref["side"]
            if ref.get("procedural_role"):
                entity["role"] = ref["procedural_role"]
            entity["canonical_name"] = ref.get("name")
            return entity

    return entity

    
def dedupe_entities(entities: List[Dict[str, Optional[str]]]) -> List[Dict[str, Optional[str]]]:
    """
    Déduplique les entités détectées.
    Si plusieurs occurrences existent, on conserve les infos les plus riches
    (rôle, camp) quand elles sont disponibles.
    """
    seen: Dict[Tuple[str, str], Dict[str, Optional[str]]] = {}
    for ent in entities:
        key = (ent["name"].lower(), ent.get("kind") or "")
        if key not in seen:
            seen[key] = dict(ent)
            continue
        existing = seen[key]
        if not existing.get("role") and ent.get("role"):
            existing["role"] = ent["role"]
        if not existing.get("side") and ent.get("side"):
            existing["side"] = ent["side"]
    return list(seen.values())


def clean_entity_name(name: str, kind: str) -> str:
    """
    Nettoie le nom brut d'une entité.
    Applique quelques règles spécifiques selon le type.
    """
    name = clean_line(name).strip(" :;,.»«\"")

    if kind == "institution":
        # On coupe ce qui suit souvent inutilement le nom de la juridiction.
        name = re.split(
            r"(?=\s+(?:que|dont|lorsque|s'est|a|aux|au|et que)\b)",
            name,
            maxsplit=1,
            flags=re.I,
        )[0]
        # On retire une éventuelle date en fin de mention.
        name = re.sub(
            r"\s+le\s+\d{1,2}\s+[a-zéûîôàè]+\s+\d{4}$",
            "",
            name,
            flags=re.I,
        )
        name = name.strip(" :;,.»«\"")

    if kind == "organization":
        # normalize casing for organization prefix
        name = re.sub(r"^(?:L['’])?ASSOCIATION\b", "Association", name, flags=re.I)

    return name


def infer_role(context: str, kind: str = "") -> Optional[str]:
    """
    Infère le rôle d'une entité à partir de son contexte proche.
    """
    if kind in ROLE_BLACKLIST_KINDS:
        return None
    short_context = context[:ROLE_WINDOW]
    for pattern, role in ROLE_HINT_PATTERNS:
        if pattern.search(short_context):
            return role
    return None


def infer_side(context: str, kind: str = "") -> Optional[str]:
    """
    Infère le camp d'une entité à partir de son contexte proche.
    """
    if kind in ROLE_BLACKLIST_KINDS:
        return None
    short_context = context[:SIDE_WINDOW]
    for pattern, side in SIDE_HINT_PATTERNS:
        if pattern.search(short_context):
            return side
    return None



def extract_entities(text: str) -> List[Dict[str, Optional[str]]]:
    """
    Détecte les entités utiles dans un texte et enrichit chaque entité avec :
    - un type,
    - un rôle supposé,
    - un camp supposé.
    """
    entities: List[Dict[str, Optional[str]]] = []
    normalized = normalize_space(text)

    for _label, pattern, default_side, kind in PERSON_ENTITY_PATTERNS:
        for match in pattern.finditer(normalized):
            raw_name = match.group(0)
            name = clean_entity_name(raw_name, kind)
            if not name:
                continue

            # On prend une petite fenêtre de contexte autour de l'entité.
            start = max(0, match.start() - 80)
            end = min(len(normalized), match.end() + 80)
            context = normalized[start:end]

            role = infer_role(context, kind=kind)
            side = infer_side(context, kind=kind) or default_side

            entities.append(
                {
                    "name": name,
                    "kind": kind,
                    "role": role,
                    "side": side,
                }
            )

    entities = [e for e in dedupe_entities(entities) if len(e["name"]) <= 120]
    entities.sort(key=lambda x: (x.get("side") is None, x.get("role") is None, x["name"]))
    return entities


    
# ============================================================
# Parsing
# ============================================================


def parse_document(text: str) -> List[SectionNode]:
    """
    Parse un document juridique en liste ordonnée de sections logiques.

    Stratégie :
    - détecter les lignes qui ressemblent à des titres,
    - construire les bornes de chaque section,
    - préserver l'ordre du document,
    - reconstruire le chemin hiérarchique via une pile,
    - enrichir chaque section avec des métadonnées.
    """
    lines = split_lines(text)

    # Extraction de l'en-tête pour pouvoir ensuite mieux interpréter les entités.
    header_registry = extract_header_registry(text)

    heading_idx: List[Tuple[int, int, str]] = []
    for i, line in enumerate(lines):
        hk = heading_level_and_kind(line)
        if hk:
            level, kind = hk
            heading_idx.append((i, level, clean_line(line)))

    # Si aucun titre n'est détecté, on considère tout le document comme un seul bloc.
    if not heading_idx:
        full = normalize_space(text)
        return [
            SectionNode(
                id="S1",
                title="DOCUMENT",
                level=1,
                start_line=0,
                end_line=len(lines) - 1,
                text=full,
                section_type="document",
                path=["DOCUMENT"],
                meta=extract_light_metadata(full, header_registry=header_registry),
            )
        ]

    nodes: List[SectionNode] = []

    # Bloc éventuel avant le premier titre : on l'interprète comme un en-tête.
    first_heading_line = heading_idx[0][0]
    if first_heading_line > 0:
        pre_text = normalize_space("\n".join(lines[:first_heading_line]))
        if pre_text:
            nodes.append(
                SectionNode(
                    id="S0",
                    title="EN-TÊTE",
                    level=1,
                    start_line=0,
                    end_line=first_heading_line - 1,
                    text=pre_text,
                    section_type="header",
                    path=["EN-TÊTE"],
                    meta=extract_light_metadata(pre_text, header_registry=header_registry),
                )
            )

    stack: List[Tuple[int, str]] = []
    inherited_type: Optional[str] = None

    for idx, (line_no, level, title) in enumerate(heading_idx):
        next_line_no = heading_idx[idx + 1][0] if idx + 1 < len(heading_idx) else len(lines)
        
        # On prend le texte depuis le titre courant jusqu'au prochain titre.
        body_lines = lines[line_no:next_line_no]
        body_text = normalize_space("\n".join(body_lines))

        # Maintien de la hiérarchie : on dépile jusqu'à retrouver un parent valide.
        while stack and stack[-1][0] >= level:
            stack.pop()
        stack.append((level, title))
        path = [item[1] for item in stack]

        # Détermine le type de section.
        section_type = guess_section_type(title, inherited=inherited_type)
        if section_type != "argument":
            inherited_type = section_type

        node = SectionNode(
            id=f"S{len(nodes) + 1}",
            title=title,
            level=level,
            start_line=line_no,
            end_line=next_line_no - 1,
            text=body_text,
            section_type=section_type,
            path=path,
            meta=extract_light_metadata(body_text, header_registry=header_registry),
        )
        nodes.append(node)

    return postprocess_nodes(nodes)



def postprocess_nodes(nodes: List[SectionNode]) -> List[SectionNode]:
    """
    Post-traitement :
    - fusionne certains micro-nœuds qui ne sont que des titres,
    - nettoie les limites de sections trop bruitées.
    """
    if not nodes:
        return nodes

    merged: List[SectionNode] = []
    i = 0
    while i < len(nodes):
        current = nodes[i]
        # Merge tiny standalone headings into next same-path/deeper-path block.
        if (
            i + 1 < len(nodes)
            and current.approx_tokens < 40
            and current.text.strip() == current.title.strip()
        ):
            nxt = nodes[i + 1]
            if nxt.path[: len(current.path)] == current.path:
                nxt.text = normalize_space(current.title + "\n\n" + nxt.text)
                nxt.path = current.path[:-1] + nxt.path[-1:]
                i += 1
                continue
        merged.append(current)
        i += 1

    # Refresh ids after merge.
    for idx, node in enumerate(merged, start=1):
        node.id = f"S{idx}"
    return merged



def extract_light_metadata(text: str, header_registry: Optional[Dict[str, object]] = None) -> Dict[str, object]:
    """
    Extrait des métadonnées légères pour un bloc de texte.
    Ces métadonnées servent à :
    - enrichir le contexte du LLM,
    - faciliter l'analyse ultérieure,
    - aider à suivre les participants et références juridiques.
    """
    articles = sorted(set(m.group(0) for m in ARTICLE_RE.finditer(text)))
    caselaw = sorted(set(clean_caselaw_match(m.group(0)) for m in CASELAW_RE.finditer(text)))
    amounts = sorted(set(m.group(0) for m in MONEY_RE.finditer(text)))
    dates = sorted(set(m.group(0) for m in DATE_RE.finditer(text)))
    entities = extract_entities(text)

    # Si on a un registre issu de l'en-tête, on tente de canoniser les entités trouvées.
    if header_registry:
        entities = [canonicalize_entity_against_registry(e, header_registry) for e in entities]

    return {
        "approx_tokens": approximate_tokens(text),
        "articles": articles[:30],
        "caselaw": caselaw[:30],
        "amounts": amounts[:30],
        "dates": dates[:30],
        "entities": entities[:20],
    }



# ============================================================
# Packeting algorithm
# ============================================================


def build_packets(
    nodes: List[SectionNode],
    max_input_tokens: int = 42000,
    prompt_budget_tokens: int = 2500,
    output_budget_tokens: int = 3000,
    min_fill_ratio: float = 0.55,
) -> List[Packet]:
    """
    Group consecutive parsed nodes into context-safe packets.

    Design goals:
    - preserve contiguous rhetorical flow,
    - prefer keeping nodes from the same parent section together,
    - split only when the safe token budget would be exceeded,
    - avoid tiny packets unless structurally necessary.
    """
    safe_content_budget = max_input_tokens - prompt_budget_tokens - output_budget_tokens
    if safe_content_budget <= 0:
        raise ValueError("Token budget too small after prompt/output reservation.")

    packets: List[Packet] = []
    current: List[SectionNode] = []
    current_tokens = 0

    def flush() -> None:
        nonlocal current, current_tokens
        if not current:
            return
        packets.append(make_packet(len(packets) + 1, current))
        current = []
        current_tokens = 0

    for idx, node in enumerate(nodes):
        node_tokens = node.approx_tokens

        if node_tokens > safe_content_budget:
            # Split oversized node recursively by paragraph groups.
            flush()
            split_nodes = split_oversized_node(node, safe_content_budget)
            for split_node in split_nodes:
                if split_node.approx_tokens > safe_content_budget:
                    raise ValueError(f"Node {split_node.id} still exceeds safe budget after splitting.")
                packets.append(make_packet(len(packets) + 1, [split_node]))
            continue

        if not current:
            current = [node]
            current_tokens = node_tokens
            continue

        same_parent = shared_parent(current[-1], node)
        would_fit = current_tokens + node_tokens <= safe_content_budget

        if would_fit:
            # Prefer continuation inside same parent / same section type.
            current.append(node)
            current_tokens += node_tokens
            continue

        # If packet is underfilled and the new node shares strong continuity,
        # try to rebalance by allowing a slightly denser packet next time.
        fill_ratio = current_tokens / safe_content_budget
        if fill_ratio < min_fill_ratio and same_parent:
            flush()
            current = [node]
            current_tokens = node_tokens
            continue

        flush()
        current = [node]
        current_tokens = node_tokens

    flush()
    return packets



def shared_parent(a: SectionNode, b: SectionNode, depth: int = 2) -> bool:
    return a.path[:depth] == b.path[:depth]



def split_oversized_node(node: SectionNode, safe_content_budget: int, header_registry: Optional[Dict[str, object]] = None) -> List[SectionNode]:
    """
    Split an oversized node by paragraph groups while preserving heading and path.
    """
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", node.text) if p.strip()]
    if len(paragraphs) <= 1:
        # Fallback: split by sentences.
        paragraphs = split_sentences(node.text)

    groups: List[List[str]] = []
    current_group: List[str] = []
    current_tokens = 0

    for para in paragraphs:
        para_tokens = approximate_tokens(para)
        if current_group and current_tokens + para_tokens > safe_content_budget:
            groups.append(current_group)
            current_group = [para]
            current_tokens = para_tokens
        else:
            current_group.append(para)
            current_tokens += para_tokens
    if current_group:
        groups.append(current_group)

    split_nodes: List[SectionNode] = []
    for i, group in enumerate(groups, start=1):
        text = normalize_space("\n\n".join(group))
        split_nodes.append(
            SectionNode(
                id=f"{node.id}_{i}",
                title=f"{node.title} [part {i}/{len(groups)}]",
                level=node.level,
                start_line=node.start_line,
                end_line=node.end_line,
                text=text,
                section_type=node.section_type,
                path=node.path + [f"part {i}/{len(groups)}"],
                meta=extract_light_metadata(text, header_registry=header_registry),
            )
        )
    return split_nodes



def split_sentences(text: str) -> List[str]:
    # Basic sentence splitter for fallback use only.
    parts = re.split(r"(?<=[\.!?;:])\s+(?=[A-ZÉÈÀÙÂÊÎÔÛÇ])", normalize_space(text))
    return [p.strip() for p in parts if p.strip()]



def make_packet(packet_num: int, nodes: List[SectionNode]) -> Packet:
    """
    Construit un objet Packet à partir d'une liste de sections.
    """
    titles = [node.title for node in nodes]
    section_types = list(dict.fromkeys(node.section_type for node in nodes))
    total_tokens = sum(node.approx_tokens for node in nodes)

    # "context_ribbon" = résumé contextuel léger utilisé pour aider le LLM
    context_ribbon = {
        "packet_id": f"P{packet_num}",
        "first_path": nodes[0].path,
        "last_path": nodes[-1].path,
        "node_ids": [node.id for node in nodes],
        "titles": titles,
        "section_types": section_types,
        "articles": sorted({a for n in nodes for a in n.meta.get("articles", [])})[:50],
        "caselaw": sorted({c for n in nodes for c in n.meta.get("caselaw", [])})[:50],
        "amounts": sorted({m for n in nodes for m in n.meta.get("amounts", [])})[:50],
        "dates": sorted({d for n in nodes for d in n.meta.get("dates", [])})[:50],
        "entities": dedupe_entities([e for n in nodes for e in n.meta.get("entities", [])])[:25],
    }

    return Packet(
        id=f"P{packet_num}",
        nodes=nodes,
        total_tokens=total_tokens,
        section_types=section_types,
        titles=titles,
        context_ribbon=context_ribbon,
    )



# ============================================================
# Prompt builders
# ============================================================


def build_extraction_system_prompt() -> str:
    """
    Prompt système pour demander à un LLM
    une extraction intermédiaire structurée sur un paquet.
    """
    return normalize_space(
        """
        Rôle
        Vous êtes un assistant juridique français chargé d'extraire et compresser fidèlement un paquet contigu de conclusions, mémoire ou plaidoirie.

        Objectif
        Produire une représentation intermédiaire structurée, riche et exploitable, sans rédiger le résumé final.
        Cette étape doit préserver au maximum l'information utile du texte source.

        Priorités
        1. Préserver le sens juridique et l'attribution des positions.
        2. Préserver la logique argumentative et l'ordre du texte.
        3. Conserver les dates, montants, articles, jurisprudences, pièces et participants mentionnés.
        4. Être concis sans supprimer les arguments substantiels.

        Règles
        - Utiliser uniquement les informations présentes dans l'entrée.
        - Ne rien inventer.
        - Si une information est incertaine, rester prudent et ne pas extrapoler.
        - Respecter l'ordre des sections et sous-sections du paquet.
        - Conserver les intitulés exacts des titres fournis.
        - Ne pas transformer ce paquet en résumé final littéraire.
        - Ne pas fusionner artificiellement des positions distinctes.
        - Si un passage reproduit la position adverse, l'indiquer explicitement.
        - Le champ "packet_summary" est uniquement un aperçu très bref.
        - Le contenu détaillé doit être conservé dans les sections.
        - Ne pas sur-comprimer les arguments à ce stade.
        - Pour chaque section importante, conserver au moins un extrait source court et fidèle.
        - Pour les sections importantes, conserver également 1 à 3 points verbatim courts lorsque cela aide à préserver la précision juridique ou factuelle.
        - Les extraits source et points verbatim doivent rester courts et strictement issus du texte fourni.
        - Identifier les participants cités dans chaque section avec leur rôle et, si possible, leur camp.
        - Si un bloc contient des prétentions finales, les isoler dans un champ dédié.

        Format de sortie
        Retourner exclusivement un JSON valide conforme au schéma demandé par l'utilisateur.
        """
    )



def build_extraction_user_prompt(packet: Packet) -> str:
    """
    Construit le prompt utilisateur pour un paquet donné.
    Le prompt inclut :
    - le contexte détecté,
    - un schéma JSON cible,
    - le texte du paquet à analyser.
    """
    schema = {
        "packet_id": packet.id,
        "document_role": "appelant|intimé|demandeur|défendeur|inconnu",
        "packet_summary": "Aperçu global très bref du paquet en 5 à 10 lignes maximum. Ce champ n'est qu'une introduction ; les détails substantiels doivent rester dans les sections.",
        "sections": [
            {
                "title": "Titre exact",
                "path": ["Niveau 1", "Niveau 2"],
                "section_type": "header|facts|procedure|discussion|claims|argument|other",
                "participants": [
                    {
                        "name": "Nom exact",
                        "kind": "person|organization|institution|lawyer|institutional_actor",
                        "role": "président|salarié|appelant|...",
                        "side": "current_party|opposing_party|neutral|null",
                    }
                ],
                "thesis": "Thèse ou fonction principale du bloc.",
                "facts": ["Fait 1", "Fait 2"],
                "arguments": ["Argument 1", "Argument 2"],
                "rebuttals": ["Réfutation 1"],
                "legal_references": ["article ...", "Cass. ..."],
                "pieces_cited": ["Pièce 17"],
                "dates_amounts": ["05 avril 2024", "30 000 €"],
                "requests": ["Demande formulée dans ce bloc"],
                "key_source_excerpt": "Court extrait source fidèle (1 à 3 phrases maximum) représentant bien le cœur du bloc.",
                "key_verbatim_points": [
                    "Point verbatim court 1",
                    "Point verbatim court 2"
                ],
                "compression_ratio_hint": "low|medium|high",
                "importance": "high|medium|low",
                "argument_density": "high|medium|low"
            }
        ],
        "carry_forward": {
            "main_issues": ["Question 1", "Question 2"],
            "open_threads": ["Point à poursuivre dans le paquet suivant"],
            "claims_block_present": True,
            "participants_to_track": [
                {
                    "name": "Nom exact",
                    "role": "fonction utile pour les paquets suivants",
                    "side": "current_party|opposing_party|neutral|null",
                }
            ],
        }
    }

    return f"""
CONTEXTE DU PAQUET
- Packet id: {packet.id}
- Types de sections: {', '.join(packet.section_types)}
- Titres inclus: {' | '.join(packet.titles)}
- Articles détectés: {', '.join(packet.context_ribbon.get('articles', [])) or 'aucun'}
- Jurisprudences détectées: {', '.join(packet.context_ribbon.get('caselaw', [])) or 'aucune'}
- Dates détectées: {', '.join(packet.context_ribbon.get('dates', [])) or 'aucune'}
- Montants détectés: {', '.join(packet.context_ribbon.get('amounts', [])) or 'aucun'}
- Participants détectés: {json.dumps(packet.context_ribbon.get('entities', []), ensure_ascii=False)}

TÂCHE
Analysez le paquet ci-dessous et retournez un JSON strictement valide.
Le JSON doit suivre cette structure cible :
{json.dumps(schema, ensure_ascii=False, indent=2)}
Consignes supplémentaires :
- "packet_summary" doit rester bref.
- Les champs des sections doivent rester substantiels.
- "key_source_excerpt" doit être court, fidèle et utile.
- "key_verbatim_points" doit contenir seulement 1 à 3 formulations exactes courtes, surtout pour les sections d'importance high.
- Ne pas multiplier les citations verbatim inutiles.

PAQUET À ANALYSER
{packet.text}
""".strip()



def build_final_system_prompt() -> str:
    """
    Prompt système utilisé dans la phase finale,
    quand tous les paquets ont déjà été extraits en JSON intermédiaire.
    """
    return normalize_space(
        """
        Rôle
        Vous êtes un assistant juridique français chargé de rédiger un résumé final exploitable à partir de représentations intermédiaires déjà extraites.

        Objectif
        Rédiger un résumé clair, fidèle, structuré et directement utile à un juriste.

        Priorités
        1. Fidélité au contenu extrait.
        2. Respect de la structure juridique utile.
        3. Exhaustivité raisonnable sur les moyens.
        4. Lisibilité professionnelle.

        Règles
        - Utiliser uniquement les données intermédiaires fournies.
        - S'appuyer en priorité sur les champs "key_source_excerpt" et "key_verbatim_points" pour préserver la précision juridique et factuelle.
        - Préserver autant que possible l'attribution des faits, arguments et demandes aux bons participants.
        - Ne pas inventer de faits, d'articles ou de demandes.
        - Préserver l'ordre général du document source.
        - Préserver les intitulés lorsque ceux-ci structurent utilement le raisonnement.
        - Les moyens doivent représenter la partie la plus substantielle du résumé.
        - Si un dispositif final a été identifié, le reproduire aussi fidèlement que possible.
        - Ne pas surcharger le texte d'avertissements méthodologiques.

        Vous pouvez produire un des rendus suivants selon la consigne utilisateur :
        - résumé global,
        - faits et procédure seulement,
        - moyens seulement,
        - rapport de synthèse,
        - exposé du litige.
        """
    )



def build_final_user_prompt(
    extracted_packets_json: List[Dict[str, object]],
    mode: str = "resume_global",
    max_pages_hint: int = 5,
) -> str:
    """
    Construit le prompt final destiné au LLM
    à partir des JSON intermédiaires produits pour chaque paquet.
    """
    mode_instructions = {
        "resume_global": "Produisez un résumé structuré complet : parties, faits, procédure, prétentions si présentes, puis moyens de façon substantielle. Lorsque les données intermédiaires contiennent des extraits source ou des points verbatim, utilisez-les comme ancrages de fidélité sans transformer le résumé en compilation de citations.",
        "faits_procedure": "Produisez uniquement les faits et la procédure, sans développer les moyens sauf mention minimale de leur objet.",
        "moyens": "Produisez uniquement la synthèse des moyens, en respectant autant que possible la structure argumentative.",
        "rapport": "Produisez un rapport de synthèse professionnel, rédigé, exploitable rapidement par un juriste.",
        "expose_litige": "Produisez un exposé du litige neutre, clair, continu et professionnel à partir des données extraites.",
    }
    selected_instruction = mode_instructions.get(mode, mode_instructions["resume_global"])

    return f"""
MODE DE SORTIE
{mode}

CONSIGNE SPÉCIFIQUE
{selected_instruction}

CONTRAINTE DE LONGUEUR
Visez un document d'environ {max_pages_hint} pages maximum à densité normale.

DONNÉES INTERMÉDIAIRES
{json.dumps(extracted_packets_json, ensure_ascii=False, indent=2)}
""".strip()


# ============================================================
# Orchestration helpers
# ============================================================


def export_parsed_nodes(nodes: List[SectionNode]) -> List[Dict[str, object]]:
    """
    Exporte les sections dans un format JSON simple.
    Utile pour inspection, debug, persistance ou intégration.
    """
    return [
        {
            "id": n.id,
            "title": n.title,
            "level": n.level,
            "path": n.path,
            "section_type": n.section_type,
            "approx_tokens": n.approx_tokens,
            "meta": n.meta,
            "text": n.text,
        }
        for n in nodes
    ]



def export_packets(packets: List[Packet]) -> List[Dict[str, object]]:
    """
    Exporte les paquets dans un format JSON simple.
    """
    return [
        {
            "id": p.id,
            "total_tokens": p.total_tokens,
            "titles": p.titles,
            "section_types": p.section_types,
            "context_ribbon": p.context_ribbon,
            "text": p.text,
        }
        for p in packets
    ]



def parse_and_packetize(
    source_text: str,
    max_input_tokens: int = 42000,
    prompt_budget_tokens: int = 2500,
    output_budget_tokens: int = 3000,
) -> Tuple[List[SectionNode], List[Packet]]:
    """
    Pipeline principal :
    1. parsing du document,
    2. découpage en paquets.
    """
    nodes = parse_document(source_text)
    packets = build_packets(
        nodes,
        max_input_tokens=max_input_tokens,
        prompt_budget_tokens=prompt_budget_tokens,
        output_budget_tokens=output_budget_tokens,
    )
    return nodes, packets


# ============================================================
# Example CLI usage
# ============================================================


def main() -> None:
    """
    Point d'entrée CLI.

    Usage typique :
    python script.py mon_document.txt --dump-dir artifacts
    """
    import argparse

    parser = argparse.ArgumentParser(description="Parse and packetize long French legal briefs.")
    parser.add_argument("input_path", type=Path, help="Path to raw txt file")
    parser.add_argument("--max-input-tokens", type=int, default=42000)
    parser.add_argument("--prompt-budget", type=int, default=2500)
    parser.add_argument("--output-budget", type=int, default=3000)
    parser.add_argument("--dump-dir", type=Path, default=Path("artifacts"))
    args = parser.parse_args()

    # Lecture du fichier source
    text = args.input_path.read_text(encoding="utf-8")

    # Parsing + packetization
    nodes, packets = parse_and_packetize(
        text,
        max_input_tokens=args.max_input_tokens,
        prompt_budget_tokens=args.prompt_budget,
        output_budget_tokens=args.output_budget,
    )

    # Création du dossier de sortie
    args.dump_dir.mkdir(parents=True, exist_ok=True)

    # Export JSON des sections
    (args.dump_dir / "parsed_nodes.json").write_text(
        json.dumps(export_parsed_nodes(nodes), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    # Export JSON des paquets
    (args.dump_dir / "packets.json").write_text(
        json.dumps(export_packets(packets), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    # Export des prompts système
    (args.dump_dir / "extraction_system_prompt.txt").write_text(
        build_extraction_system_prompt(), encoding="utf-8"
    )
    (args.dump_dir / "final_system_prompt.txt").write_text(
        build_final_system_prompt(), encoding="utf-8"
    )

    # Export d'un prompt utilisateur par paquet pour inspection / intégration LLM.
    packet_prompts_dir = args.dump_dir / "packet_prompts"
    packet_prompts_dir.mkdir(exist_ok=True)
    for packet in packets:
        (packet_prompts_dir / f"{packet.id}.txt").write_text(
            build_extraction_user_prompt(packet), encoding="utf-8"
        )

    # Petit récapitulatif terminal
    print(f"Parsed {len(nodes)} sections into {len(packets)} packets.")
    for packet in packets:
        print(f"- {packet.id}: {packet.total_tokens} approx tokens | {' | '.join(packet.titles[:4])}")

# Décommenter dans un vrai script Python hors notebook
# if __name__ == "__main__":
#    main()


## Vérification de l'extraction d'entités du header

In [160]:
text = Path("/home/jovyan/notebooks/tests_thomas/summarization/data/Dossier_6_conclusion_appelant.txt").read_text(encoding="utf-8")
registry = extract_header_registry(text)
print(json.dumps(registry, ensure_ascii=False, indent=2))


{
  "document_view": "appelant",
  "parties": [
    {
      "name": "Association [Personne Morale 1]",
      "kind": "organization",
      "procedural_role": "appelant",
      "side": "current_party",
      "lawyer": "Maître [F] [E]"
    },
    {
      "name": "Monsieur [B] [G]",
      "kind": "person",
      "procedural_role": "intimé",
      "side": "opposing_party",
      "lawyer": "Maître [B] [Q]"
    }
  ],
  "by_name": {
    "association [personne morale 1]": {
      "name": "Association [Personne Morale 1]",
      "kind": "organization",
      "procedural_role": "appelant",
      "side": "current_party",
      "lawyer": "Maître [F] [E]"
    },
    "maître [f] [e]": {
      "name": "Maître [F] [E]",
      "kind": "lawyer",
      "procedural_role": "appelant",
      "side": "current_party",
      "represents": "Association [Personne Morale 1]"
    },
    "monsieur [b] [g]": {
      "name": "Monsieur [B] [G]",
      "kind": "person",
      "procedural_role": "intimé",
      "side":

## Visualisons du parsing et de la formation des paquets

In [161]:
from pathlib import Path

text = Path("/home/jovyan/notebooks/tests_thomas/summarization/data/Dossier_6_conclusion_appelant.txt").read_text(encoding="utf-8")

nodes, packets = parse_and_packetize(text)

print(len(nodes), "sections") 
print(len(packets), "packets")

# Inspect
print("\nVisualisation des noeuds:\n")
for n in nodes[:10]:
    print("----")
    print(n.id, n.section_type)
    print(n.path)
    print(n.text[:300])

print("\nVisualisaiton des paquets:\n")
for p in packets:
    print("====", p.id, "====")
    print("tokens:", p.total_tokens)
    print("titles:", p.titles[:5])
    print("node_ids", p.context_ribbon["node_ids"][:4])
    print("articles:", p.context_ribbon["articles"])
    print("caselaw:", p.context_ribbon["caselaw"][:3])
    print("entities:", p.context_ribbon["entities"][:5])

40 sections
2 packets

Visualisation des noeuds:

----
S1 header
["CONCLUSIONS D'APPELANT"]
CONCLUSIONS D'APPELANT
 

 POUR :
 

 L'Association [Personne Morale 1] ([Personne Morale 2])
 
 , association régie par la loi du 1er juillet 1901, dont le siège social est situé [Adresse 1], prise en la personne de son représentant.
----
S2 header
['APPELANTE']
APPELANTE
 

 Maître [F] [E]
 

 CONTRE :
 

 Monsieur [B] [G], né le [Date naissance 1] 1973 à [Localité 1], de nationalité française, directeur du développement, domicilié [Adresse 2]
----
S3 header
['INTIME']
INTIME
 

 Maître [B] [Q]
----
S4 facts
['I. LES FAITS']
I. LES FAITS
 

 Monsieur [G] est entré au service de L'ASSOCIATION [Personne Morale 3] aux termes d'un contrat de travail à durée indéterminée en date du 1er septembre 2019.
 

 Monsieur [G] connaissait bien cette Association dont il était membre, il était engagé en qualité de Directeur du Développ
----
S5 procedure
['II. LA PROCEDURE']
II. LA PROCEDURE
 

 Par requête en

## Création des métadonnées et construction des prompts

In [162]:
# Build extraction prompt for one packet
system_prompt = build_extraction_system_prompt()
user_prompt = build_extraction_user_prompt(packets[0])

print("\nSYSTEM PROMPT:\n", system_prompt[:500])
print("\nUSER PROMPT:\n", user_prompt[:7000])


SYSTEM PROMPT:
 Rôle
 Vous êtes un assistant juridique français chargé d'extraire et compresser fidèlement un paquet contigu de conclusions, mémoire ou plaidoirie.

 Objectif
 Produire une représentation intermédiaire structurée, riche et exploitable, sans rédiger le résumé final.
 Cette étape doit préserver au maximum l'information utile du texte source.

 Priorités
 1. Préserver le sens juridique et l'attribution des positions.
 2. Préserver la logique argumentative et l'ordre du texte.
 3. Conserver les date

USER PROMPT:
 CONTEXTE DU PAQUET
- Packet id: P1
- Types de sections: header, facts, procedure, discussion
- Titres inclus: CONCLUSIONS D'APPELANT | APPELANTE | INTIME | I. LES FAITS | II. LA PROCEDURE | DISCUSSION | I- <u>SUR LA DEMANDE DE RESILIATION JUDICIAIRE DU</u> CONTRAT DE TRAVAIL | 1- EN DROIT | 2- EN L'ESPECE | GRIEF 1 | GRIEF 2 | GRIEF 3 | GRIEF 4 : | GRIEF 5 : | GRIEF 6 | GRIEF 7 : | GRIEF 8 | GRIEF 9 | GRIEF 10 : | GRIEF 11 | GRIEF 12 : | GRIEF 13 : | GRIEF 14 | G